In [223]:
import os, pickle
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

In [224]:
df = pd.read_csv('../data/data.csv')
stores = pd.read_csv('../data/stores.csv')
features = pd.read_csv('../data/features.csv')

In [225]:
df = df.merge(stores, how='left').merge(features, how='left')

In [226]:
def split_date(df):
    df['Date'] = pd.to_datetime(df['Date'])
    df['Year'] = df.Date.dt.year
    df['Month'] = df.Date.dt.month
    df['Day'] = df.Date.dt.day
    df['WeekOfYear'] = (df.Date.dt.isocalendar().week)*1.0  

In [227]:
split_date(df)

In [228]:
df = df.drop(['Date', 'Temperature','Fuel_Price', 'Type', 'MarkDown1', 'MarkDown2', 'MarkDown3',
             'MarkDown4', 'MarkDown5', 'CPI', 'Unemployment'], axis=1)

In [229]:
X_train = df.drop(['Weekly_Sales'], axis=1)
y_train = df['Weekly_Sales']
binary_cols = ['IsHoliday']
numeric_cols = [col for col in X_train.columns if col not in binary_cols]

In [230]:
def save_numpy_array_data(file_path: str, array: np.array):
    dir_path = os.path.dirname(file_path)
    os.makedirs(dir_path, exist_ok=True)
    with open(file_path, 'wb') as file_obj:
        np.save(file_obj, array)
    
def save_object(file_path: str, obj: Pipeline) -> None:
    os.makedirs(os.path.dirname(file_path), exist_ok=True)
    with open(file_path, "wb") as file_obj:
        pickle.dump(obj, file_obj)

def get_transformation_pipeline(binary_cols: list, numeric_cols: list) -> Pipeline:
    ft = FunctionTransformer(lambda x: x.astype(int))
    sc = StandardScaler()
    preprocessor = ColumnTransformer([
            ('binary', ft, binary_cols),
            ('numeric', sc, numeric_cols)
    ])
    pipeline = Pipeline([('preprocessor', preprocessor)])
    return pipeline

In [231]:
pipeline = get_transformation_pipeline(binary_cols=['IsHoliday'], numeric_cols=[col for col in X_train.columns if col not in ['IsHoliday']])
transformation_object = pipeline.fit(X_train)
X_train_transformed = transformation_object.transform(X_train)

In [232]:
X_train_transformed[0]

array([ 0.        , -1.65819926, -1.41874236,  0.23920895, -1.21548691,
       -1.37194493, -1.21929331, -1.47166146])

In [236]:
train_arr = np.c_[X_train_transformed, np.array(y_train)]
train_arr[0]

array([ 0.00000000e+00, -1.65819926e+00, -1.41874236e+00,  2.39208954e-01,
       -1.21548691e+00, -1.37194493e+00, -1.21929331e+00, -1.47166146e+00,
        2.49245000e+04])

In [237]:
y_train[0]

np.float64(24924.5)